In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import warnings; warnings.simplefilter('ignore')
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests
from typing import Tuple
import math

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
while not (_p / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p / 'scripts'))
from paths import P, ensure_dirs

In [ ]:
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
})
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Data loading

In [ ]:
epi_adata = sc.read_h5ad(str(P.processed.adata.epithelial / "epithelial_20_50_harmony_batch_05_pt_05_ssg.h5ad"))

res = "epithelial_0.8"

cluster_colors = {
    "0": '#009432',
    "1": '#C4E538',
    "2": "#FF788F",
    "3": "#85D5FB",
    "4": "#B3A1F9",
    "5": '#0652DD',
    "6": '#F79F1F',
    "7": "#a53707",
    "8": '#833471',
    "9": '#EA2027',
    "10": '#1B1464'
}

# Per-patient cluster composition

In [ ]:
fig_outdir = str(P.results.figures / "supplemental-clustering-per-pt")
os.makedirs(fig_outdir, exist_ok=True)

In [ ]:
def plot_cluster_composition_by_patient(
    adata,
    cluster_col: str,
    cluster_palette=None,
    figsize_per_patient: Tuple[int, int] = (6, 6),
    save_name: str = 'cluster_composition_per_patient.pdf',
):
    tissue_order = ['Dist_N', 'Adj_N', 'AD', 'CA']

    clusters = sorted(adata.obs[cluster_col].astype(str).unique(), key=lambda x: int(x))

    n_clusters = len(clusters)
    if cluster_palette is None:
        cluster_colors_local = dict(zip(clusters, sns.color_palette("tab20", n_clusters)))
    elif isinstance(cluster_palette, dict):
        cluster_colors_local = cluster_palette
        missing = [c for c in clusters if c not in cluster_colors_local]
        if missing:
            extra = sns.color_palette("tab20", len(missing))
            for c, color in zip(missing, extra):
                cluster_colors_local[c] = color
    elif isinstance(cluster_palette, list):
        if len(cluster_palette) < n_clusters:
            extra = sns.color_palette("tab20", n_clusters - len(cluster_palette))
            extra_hex = [f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}' for r, g, b in extra]
            cluster_palette = cluster_palette + extra_hex
        cluster_colors_local = dict(zip(clusters, cluster_palette[:n_clusters]))
    else:
        cluster_colors_local = cluster_palette

    tissue_counts = (
        adata.obs.groupby('patient_id')['tissue_type_cell_level_normal_split']
        .nunique()
    )
    patients = sorted(tissue_counts[tissue_counts > 1].index.tolist())
    print(f"Plotting {len(patients)} patients with >1 tissue type")
    print(f"Excluded: {sorted(tissue_counts[tissue_counts <= 1].index.tolist())}")

    ncols = 8
    nrows = math.ceil((len(patients) + 2) / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_patient[0] * ncols, figsize_per_patient[1] * nrows)
    )
    axes = axes.flatten()

    axes[0].set_visible(False)
    axes[1].set_visible(False)

    for i, patient in enumerate(patients):
        ax = axes[i + 2]
        patient_adata = adata[adata.obs['patient_id'] == patient]

        composition = pd.crosstab(
            patient_adata.obs['tissue_type_cell_level_normal_split'],
            patient_adata.obs[cluster_col].astype(str),
            normalize='index'
        ) * 100

        available = [t for t in tissue_order if t in composition.index]
        composition = composition.reindex(
            index=available,
            columns=clusters,
            fill_value=0
        )

        bottom = np.zeros(len(available))
        for cluster in clusters:
            ax.bar(
                range(len(available)),
                composition[cluster],
                bottom=bottom,
                color=cluster_colors_local[cluster],
                alpha=0.8
            )
            bottom += composition[cluster].values

        ax.set_title(patient, fontsize=10, fontweight='bold')
        ax.set_ylim(0, 100)
        ax.set_xticks(range(len(available)))
        ax.set_xticklabels(available, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Composition (%)', fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, axis='y')

    for j in range(i + 3, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        'Cluster Composition by Tissue Type - Per Patient',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig(os.path.join(fig_outdir, save_name), bbox_inches='tight')
    plt.show()

    return fig, axes


fig, axes = plot_cluster_composition_by_patient(
    adata=epi_adata,
    cluster_col='epithelial_0.8',
    cluster_palette=cluster_colors,
)

# Epithelial markers dotplot

In [ ]:
fig_outdir_epi = str(P.results.figures / "supp_11")
os.makedirs(fig_outdir_epi, exist_ok=True)

sc.settings._vector_friendly = True

sc.tl.rank_genes_groups(epi_adata, res, layer='normalized', use_raw=False)
sc.pl.rank_genes_groups_dotplot(epi_adata, n_genes=5, groupby=res,
                                title="epi_adata", standard_scale="var", show=False, dendrogram=False)
plt.savefig(os.path.join(fig_outdir_epi, "epi_dotplot.pdf"), dpi=300, bbox_inches="tight")
plt.show()

sc.settings._vector_friendly = False

# scCODA

In [ ]:
output_dir = str(P.results.figures / "figure_2")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import sccoda.util.cell_composition_data as dat
import sccoda.util.comp_ana as mod

In [ ]:
patient_counts = (
    epi_adata.obs
    .groupby(['patient_id', 'tissue_type_cell_level_normal_split'])[res]
    .value_counts()
    .unstack(fill_value=0)
)

patient_counts = patient_counts[patient_counts.sum(axis=1) > 0]
patient_counts.columns = patient_counts.columns.astype(str)

In [ ]:
patient_adata = ad.AnnData(
    X=patient_counts.values,
    obs=patient_counts.index.to_frame(index=False).rename(
        columns={patient_counts.index.names[0]: 'patient_id',
                 patient_counts.index.names[1]: 'tissue_type'}
    ),
)
patient_adata.var_names = patient_counts.columns
patient_adata.obs.index = patient_adata.obs.index.astype(str)

In [ ]:
def get_credible_series(result, tissue_level):
    ce = result.credible_effects()
    covariate_names = ce.index.get_level_values(0).unique()
    match = [c for c in covariate_names if c.endswith(f"[T.{tissue_level}]")]
    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one covariate matching level '{tissue_level}', "
            f"found {match} in {list(covariate_names)}"
        )
    sub = ce.loc[match[0]]
    sub.index = sub.index.astype(str)
    return sub

model_sep_n = mod.CompositionalAnalysis(
    patient_adata,
    formula="C(tissue_type, Treatment('Dist_N'))",
    reference_cell_type='7'
)
result_sep_n = model_sep_n.sample_hmc(num_results=20000, num_burnin=5000)
print("\n--- Reference tissue: Dist_N ---")
print(result_sep_n.credible_effects())

model_n = mod.CompositionalAnalysis(
    patient_adata,
    formula="C(tissue_type, Treatment('Adj_N'))",
    reference_cell_type='7'
)
result_n = model_n.sample_hmc(num_results=20000, num_burnin=5000)
print("\n--- Reference tissue: Adj_N ---")
print(result_n.credible_effects())

model_ta = mod.CompositionalAnalysis(
    patient_adata,
    formula="C(tissue_type, Treatment('AD'))",
    reference_cell_type='7'
)
result_ta = model_ta.sample_hmc(num_results=20000, num_burnin=5000)
print("\n--- Reference tissue: AD ---")
print(result_ta.credible_effects())

comparisons = {
    'Dist_N vs Adj_N':  get_credible_series(result_sep_n, 'Adj_N'),
    'Dist_N vs AD': get_credible_series(result_sep_n, 'AD'),
    'Dist_N vs CA': get_credible_series(result_sep_n, 'CA'),
    'Adj_N vs AD':          get_credible_series(result_n, 'AD'),
    'Adj_N vs CA':          get_credible_series(result_n, 'CA'),
    'AD vs CA':         get_credible_series(result_ta, 'CA'),
}
clusters = sorted(epi_adata.obs[res].astype(str).unique(), key=lambda x: int(x))
credible = pd.DataFrame(comparisons).loc[clusters]

if credible.isna().any().any():
    missing = credible.isna()
    raise ValueError(
        f"Missing credible-effect values detected:\n"
        f"{missing[missing.any(axis=1)]}"
    )

In [ ]:
credible.index = credible.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 6))

for i, cluster in enumerate(clusters):
    for j, comp in enumerate(credible.columns):
        if credible.loc[cluster, comp]:
            ax.scatter(j, i, s=200, c=[cluster_colors[cluster]],
                       edgecolors='black', linewidth=1, zorder=3)
        else:
            ax.scatter(j, i, s=200, facecolors='none',
                       edgecolors='lightgray', linewidth=1, zorder=3)

ax.set_yticks(range(len(clusters)))
ax.set_yticklabels(clusters, fontsize=14, fontweight='bold')
ax.set_xticks(range(len(credible.columns)))
ax.set_xticklabels(credible.columns, rotation=45, ha='right', fontsize=12)
ax.set_xlim(-0.5, len(credible.columns) - 0.5)
ax.set_ylim(-0.5, len(clusters) - 0.5)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='both', alpha=0.2)

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markersize=12, markeredgecolor='black', label='Credible'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
               markersize=12, markeredgecolor='lightgray', label='Not credible'),
]
ax.legend(handles=legend_elements, loc='upper left',
          bbox_to_anchor=(1.02, 1), frameon=True, fontsize=11)

ax.set_xlabel('Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'cluster_credibility_dotplot_patient.pdf'), bbox_inches='tight')
plt.show()

### Mann-Whitney

In [ ]:
tissue_types = ['Dist_N', 'Adj_N', 'AD', 'CA']
clusters = sorted(epi_adata.obs[res].astype(str).unique(), key=lambda x: int(x))

contingency = pd.crosstab(
    epi_adata.obs['tissue_type_cell_level_normal_split'],
    epi_adata.obs[res].astype(str)
)
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Overall chi-square: chi2={chi2:.2f}, p={p:.2e}")

pairwise_results = []
for t1, t2 in combinations(tissue_types, 2):
    sub = contingency.loc[[t1, t2]]
    chi2_pair, p_pair, _, _ = chi2_contingency(sub)
    pairwise_results.append({'tissue_1': t1, 'tissue_2': t2, 'chi2': chi2_pair, 'p': p_pair})

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df['p_adj'] = multipletests(pairwise_df['p'], method='fdr_bh')[1]
print(pairwise_df)

patient_contingency = epi_adata.obs.groupby(
    ['patient_id', 'tissue_type_cell_level_normal_split']
)[res].value_counts().unstack(fill_value=0)
patient_contingency = patient_contingency[patient_contingency.sum(axis=1) > 0]

patient_props = patient_contingency.div(patient_contingency.sum(axis=1), axis=0)
patient_props['tissue_type'] = patient_contingency.index.get_level_values(1)

cluster_results = []
for t1, t2 in combinations(tissue_types, 2):
    for cluster in clusters:
        group1 = patient_props[patient_props['tissue_type'] == t1][cluster]
        group2 = patient_props[patient_props['tissue_type'] == t2][cluster]
        if len(group1) > 0 and len(group2) > 0:
            stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
            cluster_results.append({
                'tissue_1': t1, 'tissue_2': t2,
                'cluster': cluster, 'U': stat, 'p': p
            })

cluster_df = pd.DataFrame(cluster_results)
cluster_df['p_adj'] = multipletests(cluster_df['p'], method='fdr_bh')[1]
print(cluster_df.to_string())

sig_matrix = cluster_df.pivot_table(
    index='cluster', columns=['tissue_1', 'tissue_2'], values='p_adj'
)

def sig_label(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'

sig_labels = sig_matrix.applymap(sig_label)

sig_matrix.columns = [f'{t1} vs {t2}' for t1, t2 in sig_matrix.columns]
sig_labels.columns = sig_matrix.columns

sort_order = sorted(sig_matrix.index, key=lambda x: int(x))
sig_matrix = sig_matrix.loc[sort_order]
sig_labels = sig_labels.loc[sort_order]

col_order = [
    'Dist_N vs Adj_N', 'Dist_N vs AD', 'Dist_N vs CA',
    'Adj_N vs AD', 'Adj_N vs CA', 'AD vs CA'
]
sig_matrix = sig_matrix[col_order]
sig_labels = sig_labels[col_order]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    -np.log10(sig_matrix.astype(float)),
    annot=sig_labels, fmt='', cmap='Reds',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '-log10(p_adj)'},
    ax=ax, yticklabels=False
)
ax.tick_params(axis='y', left=False)
ax.set_ylabel('')

fig.canvas.draw()
bbox = ax.get_window_extent()
ncols = len(col_order)
nrows = len(sort_order)
data_units_per_pixel_x = ncols / bbox.width
row_height_px = bbox.height / nrows
square_width = row_height_px * data_units_per_pixel_x

for i, cluster in enumerate(sort_order):
    color = cluster_colors[cluster]
    rect = mpatches.Rectangle(
        (-square_width, i), square_width, 1,
        facecolor=color, edgecolor='white', linewidth=0.5, clip_on=False
    )
    ax.add_patch(rect)
    ax.text(-square_width - 0.15, i + 0.5, cluster, ha='right', va='center',
            fontsize=14, fontweight='bold', clip_on=False)

ax.set_xlabel('Comparison', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "tissue_type_composition_mannwhitney_heatmap_patient.pdf"), bbox_inches='tight', dpi=300)
plt.show()